# Predictive-Temporal-Coding — risultati principali

**Fase:** branch `Predictive-Temporal-Coding`, sviluppo DVS-Lip. Questo notebook è una vista **read-only** di run completi identificati per ID esatto; non addestra né apre l’official test. I dati provengono da `summary.json`, `history.csv`, `prefix_curve_absolute_time.csv`, `hardware_profile_v4.json` e dalla diagnostica D. Tutti i risultati sono development-validation; i checkpoint sono stati scelti sulla stessa validation. Punteggi e intervalli fra seed sono esplorativi. Richiede `numpy`, `matplotlib` e un kernel Jupyter con `IPython`. Le celle non dipendono da `.pt`.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents) if (p / "notebooks" / "artifact_data.py").exists() and (p / "artifacts").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from this repository or one of its subdirectories")
sys.path.insert(0, str(ROOT / "notebooks"))
from artifact_data import (load_run, number, macro_f1, accuracy, group_accuracy,
                           best_train_accuracy, late_f1, deployment_parameters,
                           profile_value, physical_f1_auc, markdown_table)
plt.rcParams.update({"figure.figsize": (9, 4.8), "figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": .18})
print("Artifact root detected; exact run IDs loaded below")

fmt = lambda v, decimals=2: "—" if not np.isfinite(v) else f"{v:.{decimals}f}"


In [ ]:
RUN_IDS = {
    "C0": "dvslip_f_tcap_stage1_dwc3_d8__20260914_093427_434930__seed42",
    "C0·43": "dvslip_f_tcap_stage1_dwc3_d8__20260914_225132_054396__seed43",
    "C0·44": "dvslip_f_tcap_stage1_dwc3_d8__20260914_225137_788814__seed44",
    "L15": "dvslip_predictive_late_prefix__20260924_153414_101029__seed42",
    "D static": "dvslip_predictive_dynamic_tcap_static_control__20260927_153414_240251__seed42",
    "D dinamico": "dvslip_predictive_dynamic_tcap__20260925_225604_461593__seed42",
    "S0": "dvslip_predictive_s0__20260925_224700_220624__seed42",
}
runs = {label: load_run(ROOT, run_id, label) for label, run_id in RUN_IDS.items()}
assert all(r["summary"]["validation"]["samples"] == 2995 for r in runs.values())
print(f"Caricati {len(runs)} run; official test: nessuno")


In [ ]:
from hashlib import sha256
provenance_rows = []
for label, run in runs.items():
    profile = run["profile"]
    provenance_rows.append([
        label,
        run["summary"].get("git_commit", "—")[:12],
        sha256((run["path"] / "summary.json").read_bytes()).hexdigest()[:12],
        profile["checkpoint_sha256"][:12] if profile else "—",
    ])
display(Markdown(markdown_table(
    ["Run", "Commit run", "SHA256 summary (12)", "SHA256 checkpoint profilato (12)"],
    provenance_rows)))


## Il quadro in una tabella

F1, accuracy e Acc1 sono percentuali; **F1-AUC** integra i cinque prefissi fisici fra 250 e 2000 ms, quindi non è un test di early exit. «Ultime 16» è una media temporale, distinta dal miglior checkpoint. `—` segnala un artifact o campo assente, non uno zero. I parametri di S0 **in deployment** escludono il predittore di solo training.


In [ ]:
labels = ["C0", "L15", "D static", "D dinamico", "S0"]
fmt = lambda v, decimals=2: "—" if not np.isfinite(v) else f"{v:.{decimals}f}"
rows = []
for label in labels:
    r = runs[label]
    s = r["summary"]
    rows.append([label, str(s["seed"]), fmt(macro_f1(r)), fmt(accuracy(r)),
                 fmt(group_accuracy(r, "Acc1")), fmt(late_f1(r)),
                 fmt(best_train_accuracy(r)), fmt(physical_f1_auc(r)),
                 f'{s["trainable_parameters"]:,}', f'{deployment_parameters(r):,}'])
display(Markdown(markdown_table(
    ["Run", "Seed", "F1 best %", "Acc %", "Acc1 %", "F1 ultime 16 %",
     "Train acc best %", "F1-AUC %", "Param. training", "Param. deploy"], rows)))


In [ ]:
show = ["C0", "L15", "D static", "D dinamico", "S0"]
values = [macro_f1(runs[k]) for k in show]
colors = ["#6685a3" if k == "C0" else "#138a72" if k == "D dinamico" else "#d48032" if k.startswith("S") else "#8396a0" for k in show]
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(show[::-1], values[::-1], color=colors[::-1])
seed_range = [macro_f1(runs[k]) for k in ("C0", "C0·43", "C0·44")]
ax.axvspan(min(seed_range), max(seed_range), color="#6685a3", alpha=.13, label="Intervallo C0, seed 42–44")
for y, v in enumerate(values[::-1]): ax.text(v+.09, y, f"{v:.2f}", va="center", fontsize=9)
ax.set(xlabel="Macro-F1 best, development validation (%)", xlim=(50, max(values)+2), title="Punteggio finale: D è il segnale maggiore al seed 42")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()


## La qualità del fit e il momento della decisione

Le curve per epoca non sono indipendenti: servono a leggere convergenza e trade-off. I prefissi sono ottenuti troncando il checkpoint addestrato a 40 bin; prima di 2 s il modello opera fuori dall’orizzonte di training e il denominatore del readout cambia.


In [ ]:
tracked = ["C0", "D dinamico", "S0"]
colors = dict(zip(tracked, ["#6685a3", "#138a72", "#d48032"]))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.1))
for label in tracked:
    h = runs[label]["history"]
    x = [int(row["epoch"]) for row in h]
    axes[0].plot(x, [100*number(row["validation_macro_f1"]) for row in h], label=label, color=colors[label], lw=1.7)
    axes[1].plot(x, [100*number(row["train_accuracy"]) for row in h], label=label, color=colors[label], lw=1.7)
axes[0].set(title="Validation Macro-F1", xlabel="Epoca", ylabel="%")
axes[1].set(title="Accuracy sul training", xlabel="Epoca", ylabel="%")
axes[0].legend(fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 4.8))
for label in tracked:
    points = sorted((int(row["requested_us"])/1e6, 100*number(row["macro_f1"])) for row in runs[label]["prefix"])
    if points:
        ax.plot(*zip(*points), marker="o", lw=2, label=label, color=colors[label])
ax.axvline(1.5, color="gray", ls="--", lw=1)
ax.set(title="Informazione disponibile al prefisso fisico", xlabel="Durata processata (s)", ylabel="Macro-F1 (%)", xticks=[.25,.5,1,1.5,2])
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


## Meccanismo S0

La skill positiva del predittore non implica beneficio sulla classificazione. `VΔ` misura la variazione temporale delle feature stage2 nella regione attiva. La curva permette di confrontare l'apprendimento della predizione con l'evoluzione delle feature.


In [ ]:
family = ["S0"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.1))
for label in family:
    h = runs[label]["history"]
    x = [int(row["epoch"]) for row in h]
    for ax, key in zip(axes, ["validation_temporal_variation_stage2_active", "validation_stage2_temporal_prediction_active_skill_vs_persistence"]):
        if key in h[0]: ax.plot(x, [number(row.get(key)) for row in h], lw=1.7, label=label)
axes[0].set(title="VΔ: dinamica delle feature", xlabel="Epoca", ylabel="VΔ stage2 attiva")
axes[1].set(title="Skill predittiva rispetto alla persistenza", xlabel="Epoca", ylabel="Skill validation")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()


## D: due controlli, due domande diverse

**D static** è stato *riaddestrato*: verifica se la dipendenza dall’input aiuta rispetto a un gate statico. Il controllo *checkpoint-only* sostituisce i gate di D con la loro media senza riaddestrare: misura dipendenza interna del checkpoint, non la prestazione ottimale di un modello statico. Non scambiare i due delta.


In [ ]:
diag_path = runs["D dinamico"]["path"] / "dynamic_routing_diagnostic.json"
import json
diag = json.loads(diag_path.read_text(encoding="utf-8"))
control_labels = ["D static\nriaddestrato", "D dinamico", "D con gate medi\ncheckpoint-only"]
control_values = [macro_f1(runs["D static"]), macro_f1(runs["D dinamico"]), 100*diag["train_mean_constant"]["macro_f1"]]
fig, ax = plt.subplots(figsize=(8.5, 4))
ax.bar(control_labels, control_values, color=["#8396a0", "#138a72", "#d5b1a9"])
for i, v in enumerate(control_values): ax.text(i, v+.5, f"{v:.2f}%", ha="center")
ax.set(ylabel="Macro-F1 validation (%)", title="Routing appreso: ablation con e senza riaddestramento", ylim=(0, max(control_values)+9))
plt.tight_layout(); plt.show()


In [ ]:
profile_labels = ["C0", "D static", "D dinamico", "S0"]
rows = []
for label in profile_labels:
    r = runs[label]
    rows.append([label, "sì" if r["profile"] else "no",
                 f'{deployment_parameters(r):,}',
                 fmt(profile_value(r,"activity","global_firing_rate"),4),
                 fmt(profile_value(r,"operations_per_sample","sop_potential")/1e9,3),
                 fmt(profile_value(r,"operations_per_sample","multivalued_mac_potential")/1e9,3),
                 fmt(profile_value(r,"energy_reference","covered_arithmetic_activity_proxy_uj_per_sample"),1)])
display(Markdown(markdown_table(["Run", "Profilato", "Param. deploy", "Firing", "SOP pot. G", "MAC multiv. G", "Horowitz att. µJ"], rows)))


**Lettura:** D guadagna sul controllo statico addestrato; il suo vantaggio resta da verificare su altri seed. S0 conserva skill predittiva, ma peggiora il fit discriminativo e il prefisso a 1,5 s. Il suo predittore di solo training non compare nel profilo di deployment. La proxy Horowitz copre solo aritmetica FP32 selezionata (45 nm), non energia totale, memoria, controllo o FPGA. I profili sono sul best checkpoint e su un sottoinsieme di validation; il profilo di C0 seed 42 manca e resta `—`, senza imputazioni da altri seed.
